# Nemotron-3-Nano-30B-A3B @ DeepInfra

This document provides a tutorial on NVIDIA's newest Nemotron 3 Nano model hosted by DeepInfra. Below you'll find information about how to use it with public/private endpoints and run benchmarks.

## Table of Contents

- [0. Pricing](#0-pricing-at-deepinfra)
- [1. Setup](#1-setup)
- [2. Basic Inference](#2-basic-inference)
- [3. Reasoning Parameters](#3-reasoning-parameters)
- [4. Tool calling abilities](#4-tool-calling-abilities)
- [5. Setting up private instance](#5-setting-up-private-instance-with-deepinfra)
- [6. Benchmarks](#6-benchmarks)


<a id="0-pricing-at-deepinfra"></a>
## 0. Pricing at DeepInfra

DeepInfra offers competitive pricing for Nemotron 3 Nano. Here is the pricing breakdown:

### Public Endpoint

- **Input tokens**: $0.06 per million tokens
- **Output tokens**: $0.24 per million tokens
- **Best for**: Development, testing, and low-to-medium volume workloads
- **Benefits**: Pay only for what you use, no setup required

### Private Endpoints

Private endpoints are priced per GPU hour and provide dedicated resources.

**1x H100-80GB (recommended)**
- **Cost**: $1.69 per hour
- **Performance**: Up to 150 tokens per second
- **Best for**: High-throughput production workloads

**1x A100-80GB**
- **Cost**: $0.89 per hour
- **Best for**: Cost-effective private hosting with good performance

**Note:** Pricing is subject to change. For the most current pricing information, visit https://deepinfra.com/nvidia/Nemotron-3-Nano-30B-A3B

<a id="1-setup"></a>
## 1. Setup

Follow these steps to set up your environment for using Nemotron 3 Nano on DeepInfra.


### Step 1: Get Your API Keys

1. Get your DeepInfra API key from: https://deepinfra.com/dash/keys
2. (Optional) Get your Hugging Face token from: https://huggingface.co/settings/tokens (needed for downloading benchmark datasets)

### Step 2: Create Environment File

Create a `.env` file in your project directory with your API keys:

```bash
cat > .env << EOF
DEEPINFRA_API_KEY=your-deepinfra-api-key-here
HF_TOKEN=your-huggingface-api-key
EOF
```

### Step 3: Install Dependencies

Install the required Python packages for this tutorial:

In [ ]:
!pip install openai python-dotenv huggingface-hub datasets

<a id="2-basic-inference"></a>
## 2. Basic Inference

Let's start with a simple inference example to verify the setup. DeepInfra supports OpenAI-compatible chat completion endpoints, so you can use the OpenAI Python package for inference.


In [ ]:
import os

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(api_key=os.getenv("DEEPINFRA_API_KEY"), base_url="https://api.deepinfra.com/v1")

response = client.chat.completions.create(
    model="nvidia/Nemotron-3-Nano-30B-A3B",
    messages=[
        {"role": "system", "content": "You are a helpful assistant named Temirulan."},
        {"role": "user", "content": "Hello, how are you? What is your name?"}
    ],
    temperature=0.7
)

print(f"Reasoning:\n{response.choices[0].message.reasoning_content}")
print(f"Answer:\n{response.choices[0].message.content}")


OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

<a id="3-reasoning-parameters"></a>
## 3. Reasoning Parameters

As you can see in the example above, the model performs reasoning by default. The Nemotron family supports controlling reasoning behavior through the `reasoning_effort` parameter.

### When to Use Reasoning

- **Enable reasoning** (default): Best for complex problems, mathematical calculations, and tasks requiring step-by-step thinking
- **Disable reasoning**: Optimal for speed when you need fast responses for simple queries or when reasoning isn't necessary

### Disabling Reasoning

You can disable reasoning by setting `reasoning_effort: 'none'` in the `extra_body` parameter. This will result in faster inference with no reasoning content:


In [ ]:
response = client.chat.completions.create(
    model="nvidia/Nemotron-3-Nano-30B-A3B",
    messages=[
        {"role": "user", "content": "If Alice is twice as old as Bob, and Bob is 5 years older than Carol, and Carol is 10, how old is Alice?"}
    ],
    temperature=0.0,
    extra_body={
        'reasoning_effort': 'none'
    }
)
print(f"Reasoning:\n{'='*100}\n{response.choices[0].message.reasoning_content}\n{'='*100}") # should be empty
print(f"Answer:\n{'='*100}\n{response.choices[0].message.content}\n{'='*100}")


Reasoning:

Answer:
We are given the following relationships:

- Carol is 10 years old.  
- Bob is 5 years older than Carol.  
- Alice is twice as old as Bob.

---

**Step 1: Find Bob’s age**  
Bob is 5 years older than Carol:  
Bob = Carol + 5 = 10 + 5 = **15**

**Step 2: Find Alice’s age**  
Alice is twice as old as Bob:  
Alice = 2 × Bob = 2 × 15 = **30**

---

✅ **Answer: Alice is 30 years old.**


<a id="4-tool-calling-abilities"></a>
## 4. Tool Calling Abilities

The Nemotron 3 Nano family supports tool calling. Below you can find a basic usage example:


In [ ]:
TOOLS = [
    {"type": "function", "function": {
        "name": "get_stock_price",
        "description": "Get the current stock price",
        "parameters": {"type": "object", "properties": {"symbol": {"type": "string", "description": "The stock symbol"}}, "additionalProperties": False, "required": ["symbol"]},
        "strict": True
    }},
    {"type": "function", "function": {
        "name": "get_weather",
        "description": "Determine weather in my location",
        "parameters": {"type": "object", "properties": {"location": {"type": "string", "description": "The city and state e.g. San Francisco, CA"},"unit": {"type": "string", "enum": ["c", "f"]}}, "additionalProperties": False, "required": ["location", "unit"]},
        "strict": True
    }}
]
response = client.chat.completions.create(
    model="nvidia/Nemotron-3-Nano-30B-A3B",
    messages=[{"role": "user", "content": "What is the weather in Tokyo on 2023-12-11?"}],
    temperature=0.0,
    tools=TOOLS
)

print(f"Reasoning:\n{'='*100}\n{response.choices[0].message.reasoning_content}\n{'='*100}")
print(f"Tool calls:\n{'='*100}\n{response.choices[0].message.tool_calls}\n{'='*100}")
# The model replies which tool call should be invoked

Reasoning:
Okay, the user is asking about the weather in Tokyo on December 11, 2023. Let me check the tools available. There's a get_weather function that requires a location and a unit. The location should be a city and state, like "San Francisco, CA". The user specified Tokyo, so I need to format that as "Tokyo, Japan" maybe? Wait, the example given was "San Francisco, CA", so maybe just "Tokyo, Japan" would work. The unit parameter is required, and it's an enum of "c" or "f". The user didn't specify Celsius or Fahrenheit, so I should probably default to one. Since the user is asking in Tokyo, which commonly uses Celsius, maybe I should use "c" as the unit. But wait, the function requires both location and unit. The user didn't mention the unit, so I need to choose. Let me check the function parameters again. The unit is required, so I can't omit it. The user's question doesn't specify, so I should pick one. Maybe the function expects the user to provide it, but since the user didn't

The model also handles more complex requests, including chat history with previously called tools and new tool requests on top of that:

In [ ]:
MESSAGE_HISTORY = [
    {"role": "system", "content": "You're an assistant named Tom"},
    {"role": "user", "content": "Hi, I'm Juan"},
    {"role": "assistant", "content": "Hi Juan! I'm Tom, your assistant. How can I help you today?"},
    {"role": "user", "content": "What's the weather like in Saint-Petersburg?"},
    {
        "role": "assistant", 
        "content": "", 
        "tool_calls": [{"id": "call_oMwI5SmMWr8HElvm8Fq472hv", "type": "function", "function": {"name": "get_weather", "arguments": "{\"location\":\"Saint-Petersburg\",\"unit\":\"c\"}"}}]
    },
    {
        "role": "tool", 
        "content": """{
            "location": "Saint-Petersburg", 
            "unit": "c", 
            "temperature": 15,
            "condition": "Partly Cloudy", 
            "humidity": 78, 
            "wind_speed": 10,
            "forecast": [
                { "day": "Monday", "high": 18, "low": 12, "condition": "Mostly Sunny" },
                { "day": "Tuesday", "high": 20, "low": 13, "condition": "Light Rain" },
                { "day": "Wednesday", "high": 19, "low": 11, "condition": "Overcast" }
            ]
        }""", 
        "tool_call_id": "call_oMwI5SmMWr8HElvm8Fq472hv"
    },
    {
        "role": "assistant", 
        "content": "Right now in Saint-Petersburg, it's partly cloudy with a temperature of 15°C. The humidity is 78%, and there's a light wind at 10 km/h.\n\nIf you'd like a forecast for the next few days, just let me know!"
    },
    {"role": "user", "content": "What is the stock price of AAPL?"}
]

response = client.chat.completions.create(
    model="nvidia/Nemotron-3-Nano-30B-A3B",
    messages=MESSAGE_HISTORY,
    temperature=0.0,
    tools=TOOLS
)

print(f"Reasoning:\n{'='*100}\n{response.choices[0].message.reasoning_content}\n{'='*100}")
print(f"Tool calls:\n{'='*100}\n{response.choices[0].message.tool_calls}\n{'='*100}")


Reasoning:
We need to get stock price of AAPL. Use get_stock_price tool with symbol "AAPL". Then respond with price.

Tool calls:
[ChatCompletionMessageToolCall(id='call_77e7c23eec4db87e', function=Function(arguments='{"symbol": "AAPL"}', name='get_stock_price'), type='function')]


<a id="5-setting-up-private-instance-with-deepinfra"></a>
## 5. Setting Up Private Instance with DeepInfra

With DeepInfra you can also set up a private instance to avoid running your inferences on the public endpoint. Having a private instance will give you full GPU utilization. You can find documentation here: https://deepinfra.com/docs/advanced/custom_llms.

Here is an example of how to deploy a private instance for this model using the CLI:

```bash
curl -X POST https://api.deepinfra.com/deploy/llm -d '{
    "model_name": "Nemotron-3-Nano-30B-A3B",
    "base_model": "nvidia/Nemotron-3-Nano-30B-A3B",
    "gpu": "H100-80GB",
    "num_gpus": 1,
    "max_batch_size": 64,
    "settings": {
        "min_instances": 1,
        "max_instances": 1
    }
}' -H 'Content-Type: application/json' \
    -H 'Authorization: Bearer $DEEPINFRA_TOKEN'
# the response would be something like:
{
    "type":"llm",
    "deploy_id":"DUxCyxDEO7Ujgm9i",
    "model_name":"Temirulan/Nemotron-3-Nano-30B-A3B",
    ...
}
```

The command above will start deploying a new instance. The `deploy_id` field is important. Using that value, you can check the status of the deployment:

```bash
curl https://api.deepinfra.com/deploy/DUxCyxDEO7Ujgm9i -H 'Content-Type: application/json' -H 'Authorization: Bearer $DEEPINFRA_TOKEN'

# the response:
{
    "type":"llm",
    "deploy_id":"DUxCyxDEO7Ujgm9i",
    "model_name":"Temirulan/Nemotron-3-Nano-30B-A3B",
    "status":"OK",
    ...
}
```

After the deployment finishes successfully, your private endpoint will be available under the model name: `{YOUR_DISPLAY_NAME}/Nemotron-3-Nano-30B-A3B`. You can use all the scripts above with this model name.

<a id="6-benchmarks"></a>
## 6. Benchmarks

In this section we will show how to run a GPQA benchmark evaluation.

[GPQA (Graduate-Level Google-Proof Q&A)](https://huggingface.co/datasets/Idavidrein/gpqa) is a multiple-choice dataset where:
* Expert accuracy: PhD-level experts in their domain achieve 65-74% accuracy
* Non-expert accuracy: Highly skilled validators outside the domain only reach 34% despite 30+ minutes with Google
* 198 questions in GPQA Diamond (the highest quality subset)
* Domains: Biology, Physics, and Chemistry

### Download GPQA Dataset from Hugging Face

Before running, you need to accept the dataset terms in the Hugging Face link above.

In [ ]:
from datasets import load_dataset
from dotenv import load_dotenv
from openai import OpenAI

client = OpenAI(
    base_url="https://api.deepinfra.com/inference/v1",
    api_key=os.getenv('DEEPINFRA_API_KEY'),
)

load_dotenv()

HF_TOKEN = os.getenv('HF_TOKEN')

dataset = load_dataset("Idavidrein/gpqa", "gpqa_diamond")
gpqa_questions = dataset['train']

print(f"✓ Successfully loaded GPQA Diamond dataset")
print(f"  Total questions: {len(gpqa_questions)}")


/home/temirulan/miniconda3/envs/di-main/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Successfully loaded GPQA Diamond dataset
  Total questions: 198


In [ ]:
benchmark_models = {
    "nemotron-3-nano-30b-a3b": {
        "model": "nvidia/Nemotron-3-Nano-30B-A3B", # replace with your model name if you want to test a private endpoint
        "display_name": "Nemotron 3 Nano 30B A3B",
    }
}

In [ ]:
# preparing helper functions to query models

import os
import json
import re
import random
from datetime import datetime
from pathlib import Path
from openai import OpenAI
import concurrent.futures
import threading

def parse_answer(response_text):
    """Parse the model's response to extract the answer letter."""
    lines = response_text.strip().split('\n')
    # FIRST: Try strict last line parsing (current method)
    if lines:
        last_line = lines[-1].strip()
        answer_match = re.search(r'ANSWER:\s*([A-D])', last_line, re.IGNORECASE)
        if answer_match:
            return answer_match.group(1).upper()
    
    # FALLBACK: Check last 3 lines with LaTeX formatting removal
    last_lines = lines[-3:] if len(lines) >= 3 else lines
    for line in reversed(last_lines):
        line_clean = line.strip()
        # Remove LaTeX formatting: $$, $, \text{...}
        line_clean = line_clean.replace('$$', '').replace('$', '')
        line_clean = re.sub(r'\\text\{([^}]*)\}', r'\1', line_clean)
        line_clean = line_clean.strip()
        # Look for "ANSWER: X" pattern (case insensitive)
        answer_match = re.search(r'ANSWER:\s*([A-D])', line_clean, re.IGNORECASE)
        if answer_match:
            return answer_match.group(1).upper()
    
    # ADDITIONAL FALLBACK: Look for answer patterns anywhere in the text
    # Check for boxed answers like \boxed{B} or Answer: B
    boxed_match = re.search(r'\\boxed\{([A-D])\}', response_text, re.IGNORECASE)
    if boxed_match:
        return boxed_match.group(1).upper()
    
    # Check for "Answer: X" pattern anywhere in text
    answer_pattern = re.search(r'Answer:\s*([A-D])', response_text, re.IGNORECASE)
    if answer_pattern:
        return answer_pattern.group(1).upper()
    
    # Check for standalone letter answers in parentheses or brackets
    standalone_match = re.search(r'[\(\[]([A-D])[\)\]]', response_text)
    if standalone_match:
        return standalone_match.group(1).upper()
    
    # If not found in any method, return None
    return None
def evaluate_model_on_gpqa(model_name, questions, max_tokens=262144, temperature=0.1, sample_size=None, random_seed=42, disable_reasoning=False, concurrent_requests=1):
    """
    Evaluate a model on GPQA questions using standard benchmark prompt format.
    Args:
        model_name: Clean model name for log file (e.g., 'nemotron_9b')
        questions: GPQA dataset questions
        max_tokens: Maximum tokens for response
        temperature: Sampling temperature (low for reasoning tasks)
        sample_size: Optional - evaluate on subset of questions
        random_seed: Seed for randomizing answer positions
        concurrent_requests: Number of parallel requests to make (default: 1)
    
    Returns:
        Dictionary with results and accuracy metrics
    """
    client = OpenAI(
        base_url="https://api.deepinfra.com/v1",
        api_key=os.getenv('DEEPINFRA_API_KEY'),
    )
    # Set random seed for reproducibility
    random.seed(random_seed)
    # Create results/gpqa directory if it doesn't exist
    log_dir = Path("benchmark/results/gpqa")
    log_dir.mkdir(parents=True, exist_ok=True)
    # Generate log filename with timestamp
    # Sanitize model name to avoid directory issues
    safe_model_name = model_name.replace('/', '_')
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_file = log_dir / f"gpqa_{safe_model_name}_{timestamp}.jsonl"
    print(f"  Logging to: {log_file}")
    if sample_size and sample_size < len(questions):
        indices = random.sample(range(len(questions)), sample_size)
        questions_to_eval = [questions[i] for i in indices]
        print(f"  Evaluating on {sample_size} randomly sampled questions")
    else:
        questions_to_eval = questions
        print(f"  Evaluating on all {len(questions)} questions")
    
    results = []
    correct = 0
    parsing_failures = 0
    lock = threading.Lock()
    completed = 0
    
    def process_question(args):
        i, q = args
        nonlocal correct, parsing_failures, completed
        
        # Use local random seeded with global seed + index for reproducibility in threads
        local_random = random.Random(random_seed + i)
        
        # Create list of all answers with labels
        all_answers = [
            ("Correct", q['Correct Answer']),
            ("Incorrect1", q['Incorrect Answer 1']),
            ("Incorrect2", q['Incorrect Answer 2']),
            ("Incorrect3", q['Incorrect Answer 3'])
        ]
        # Shuffle the answers
        local_random.shuffle(all_answers)
        # Map shuffled answers to A, B, C, D
        answer_mapping = {
            'A': all_answers[0],
            'B': all_answers[1],
            'C': all_answers[2],
            'D': all_answers[3]
        }
        # Find which letter corresponds to the correct answer
        correct_letter = None
        for letter, (label, text) in answer_mapping.items():
            if label == "Correct":
                correct_letter = letter
                break
        
        # Format question with GPQA standard prompt format
        instruction = """Answer the following multiple choice question. The last line of your response should be of the following format: 'ANSWER: $LETTER' (without quotes) where LETTER is one of A,B,C,D. Think step by step before answering.

"""
        
        question_text = instruction
        question_text += f"{q['Question']}\n\n"
        question_text += f"A) {answer_mapping['A'][1]}\n"
        question_text += f"B) {answer_mapping['B'][1]}\n"
        question_text += f"C) {answer_mapping['C'][1]}\n"
        question_text += f"D) {answer_mapping['D'][1]}"
        
        # Build the messages array that will be sent to the model
        messages = [{"role": "user", "content": question_text}]
        
        try:
            extra_body = {}
            if disable_reasoning:
                extra_body['reasoning_effort'] = 'none'
            
            # Using model_name instead of hardcoded string
            response = client.chat.completions.create(
                model=model_name,
                messages=messages,
                max_tokens=max_tokens,
                temperature=temperature,
                extra_body=extra_body
            )
            full_response = response.choices[0].message.content.strip()
            # Check if response was likely cut off (doesn't end with ANSWER: pattern)
            finish_reason = response.choices[0].finish_reason
            response_cutoff = finish_reason == "length"
            
            # Parse the answer from the response (strict parsing)
            parsed_answer = parse_answer(full_response)
            
            with lock:
                # Track if parsing failed
                if parsed_answer is None:
                    parsing_failures += 1
                    if response_cutoff:
                        print(f"  Warning: Question {i+1} - Response cut off at max_tokens limit")
                    else:
                        print(f"  Warning: Question {i+1} - Failed to parse answer (model didn't follow format) |{full_response}|")
                
                # Check if answer is correct
                is_correct = parsed_answer == correct_letter if parsed_answer else False
                if is_correct:
                    correct += 1
                
                result_entry = {
                    "question_index": i,
                    "subdomain": q['Subdomain'],
                    "question": q['Question'],
                    "choices": {
                        "A": answer_mapping['A'][1],
                        "B": answer_mapping['B'][1],
                        "C": answer_mapping['C'][1],
                        "D": answer_mapping['D'][1]
                    },
                    "correct_answer": correct_letter,
                    "answer_mapping": {k: v[0] for k, v in answer_mapping.items()},
                    "model_input": messages,
                    "model_full_response": full_response,
                    "model_parsed_answer": parsed_answer,
                    "parsing_failed": parsed_answer is None,
                    "response_cutoff": response_cutoff,
                    "finish_reason": finish_reason,
                    "is_correct": is_correct,
                    "temperature": temperature,
                    "max_tokens": None,
                    "random_seed": random_seed,
                    "timestamp": datetime.now().isoformat()
                }
                
                results.append({
                    "subdomain": q['Subdomain'],
                    "answer": parsed_answer,
                    "is_correct": is_correct
                })
                
                # Log to file immediately
                with open(log_file, 'a') as f:
                    f.write(json.dumps(result_entry) + '\n')
                
                completed += 1
                # Progress update every 10 questions
                if completed % 10 == 0:
                    current_acc = (correct / completed) * 100
                    print(f"  Progress: {completed}/{len(questions_to_eval)} | Current accuracy: {current_acc:.1f}% | Parse failures: {parsing_failures}")
                
        except Exception as e:
            with lock:
                print(f"  Error on question {i+1}: {e}")
                import traceback
                traceback.print_exc()
                error_entry = {
                    "question_index": i,
                    "subdomain": q['Subdomain'],
                    "question": q['Question'],
                    "model_input": messages,
                    "error": str(e),
                    "is_correct": False,
                    "timestamp": datetime.now().isoformat()
                }
                
                results.append({
                    "subdomain": q['Subdomain'],
                    "answer": None,
                    "is_correct": False
                })
                
                with open(log_file, 'a') as f:
                    f.write(json.dumps(error_entry) + '\n')
                completed += 1

    # Run requests in parallel
    print(f"  Starting evaluation with {concurrent_requests} concurrent requests...")
    with concurrent.futures.ThreadPoolExecutor(max_workers=concurrent_requests) as executor:
        list(executor.map(process_question, enumerate(questions_to_eval)))
    
    accuracy = (correct / len(results)) * 100 if results else 0
    parsing_failure_rate = (parsing_failures / len(results)) * 100 if results else 0
    
    # Write summary at the end of log file
    summary = {
        "summary": True,
        "model_name": model_name,
        "total_questions": len(results),
        "correct": correct,
        "accuracy": accuracy,
        "parsing_failures": parsing_failures,
        "parsing_failure_rate": parsing_failure_rate,
        "sample_size": sample_size,
        "temperature": temperature,
        "max_tokens": max_tokens,
        "random_seed": random_seed,
        "concurrent_requests": concurrent_requests,
        "prompt_format": "GPQA standard (strict last-line parsing, randomized positions)",
        "timestamp": datetime.now().isoformat()
    }
    
    with open(log_file, 'a') as f:
        f.write(json.dumps(summary) + '\n')
    
    print(f"  ✓ Results logged to: {log_file}")
    print(f"  ✓ Parsing failures: {parsing_failures}/{len(results)} ({parsing_failure_rate:.1f}%)")
    
    return {
        'results': results,
        'accuracy': accuracy,
        'correct': correct,
        'total': len(results),
        'parsing_failures': parsing_failures,
        'log_file': str(log_file)
    }

print("✓ GPQA evaluation function ready (with strict answer parsing and high token limit)")


✓ GPQA evaluation function ready (with strict answer parsing and high token limit)


### Running Benchmarks

Control the number of test questions with the `SAMPLE_SIZE` variable. Set it to `None` to run on all questions.

In [ ]:
# Evaluate Nemotron 9B
# ============================================================================
SAMPLE_SIZE = None
CONCURRENT_REQUESTS = 40

model_key = 'nemotron-3-nano-30b-a3b'
model_config = benchmark_models[model_key]

print(f"\n{'='*80}")
print(f"Evaluating: {model_config['display_name']}")
print(f"{'='*80}")

try:
    result = evaluate_model_on_gpqa(
        model_config['model'],
        gpqa_questions,
        sample_size=SAMPLE_SIZE,
        concurrent_requests=CONCURRENT_REQUESTS
    )
    
    print(f"\n✓ {model_config['display_name']} Results:")
    print(f"  Accuracy: {result['accuracy']:.2f}%")
    print(f"  Correct: {result['correct']}/{result['total']}")
    print(f"  Parsing failures: {result['parsing_failures']}")
    print(f"  Log file: {result['log_file']}")
    
except Exception as e:
    print(f"\n✗ Error evaluating {model_config['display_name']}: {e}")



Evaluating: Nemotron Nano 3 30B A3B
  Logging to: benchmark/results/gpqa/gpqa_nvidia_Nemotron-Nano-3-30B-A3B_20251211_213459.jsonl
  Evaluating on all 198 questions
  Starting evaluation with 40 concurrent requests...
  Progress: 10/198 | Current accuracy: 100.0% | Parse failures: 0
  Progress: 20/198 | Current accuracy: 95.0% | Parse failures: 0
  Progress: 30/198 | Current accuracy: 66.7% | Parse failures: 8
  Progress: 40/198 | Current accuracy: 50.0% | Parse failures: 18
  Progress: 50/198 | Current accuracy: 46.0% | Parse failures: 25
  Progress: 60/198 | Current accuracy: 46.7% | Parse failures: 30
  Progress: 70/198 | Current accuracy: 48.6% | Parse failures: 34
  Progress: 80/198 | Current accuracy: 45.0% | Parse failures: 41
  Progress: 90/198 | Current accuracy: 40.0% | Parse failures: 51
  Progress: 100/198 | Current accuracy: 37.0% | Parse failures: 60
  Progress: 110/198 | Current accuracy: 38.2% | Parse failures: 65
  Progress: 120/198 | Current accuracy: 39.2% | Parse f

In [ ]:
# Evaluate Nemotron 9B v2
# ============================================================================
SAMPLE_SIZE = None
CONCURRENT_REQUESTS = 40

model_key = 'nemotron-nano-9b-v2'

print(f"\n{'='*80}")
print(f"Evaluating: nemotron-nano-9b-v2")
print(f"{'='*80}")

try:
    result = evaluate_model_on_gpqa(
        "nvidia/NVIDIA-Nemotron-Nano-9B-v2",
        gpqa_questions,
        sample_size=SAMPLE_SIZE,
        concurrent_requests=CONCURRENT_REQUESTS
    )
    
    print(f"\n✓ nemotron-nano-9b-v2 Results:")
    print(f"  Accuracy: {result['accuracy']:.2f}%")
    print(f"  Correct: {result['correct']}/{result['total']}")
    print(f"  Parsing failures: {result['parsing_failures']}")
    print(f"  Log file: {result['log_file']}")
    
except Exception as e:
    print(f"\n✗ Error evaluating nemotron-nano-9b-v2: {e}")



Evaluating: nemotron-nano-9b-v2
  Logging to: benchmark/results/gpqa/gpqa_nvidia_NVIDIA-Nemotron-Nano-9B-v2_20251211_214752.jsonl
  Evaluating on all 198 questions
  Starting evaluation with 40 concurrent requests...
  Progress: 10/198 | Current accuracy: 70.0% | Parse failures: 0
  Progress: 20/198 | Current accuracy: 70.0% | Parse failures: 0
  Progress: 30/198 | Current accuracy: 66.7% | Parse failures: 0
  Progress: 40/198 | Current accuracy: 67.5% | Parse failures: 0


In [ ]:
# Evaluate gemma 3 27b
# ============================================================================
SAMPLE_SIZE = None
CONCURRENT_REQUESTS = 40

model_key = 'gemma-3-27b-it'

print(f"\n{'='*80}")
print(f"Evaluating: gemma-3-27b-it")
print(f"{'='*80}")

try:
    result = evaluate_model_on_gpqa(
        "google/gemma-3-27b-it",
        gpqa_questions,
        sample_size=SAMPLE_SIZE,
        concurrent_requests=CONCURRENT_REQUESTS
    )
    
    print(f"\n✓ gemma-3-27b-it Results:")
    print(f"  Accuracy: {result['accuracy']:.2f}%")
    print(f"  Correct: {result['correct']}/{result['total']}")
    print(f"  Parsing failures: {result['parsing_failures']}")
    print(f"  Log file: {result['log_file']}")
    
except Exception as e:
    print(f"\n✗ Error evaluating gemma-3-27b-it: {e}")


In [ ]:
# qwen3 30b a3b
# ============================================================================
SAMPLE_SIZE = None
CONCURRENT_REQUESTS = 40

model_key = 'Qwen/Qwen3-30B-A3B'

print(f"\n{'='*80}")
print(f"Evaluating: Qwen/Qwen3-30B-A3B")
print(f"{'='*80}")

try:
    result = evaluate_model_on_gpqa(
        "Qwen/Qwen3-30B-A3B",
        gpqa_questions,
        sample_size=SAMPLE_SIZE,
        concurrent_requests=CONCURRENT_REQUESTS
    )
    
    print(f"\n✓ Qwen/Qwen3-30B-A3B Results:")
    print(f"  Accuracy: {result['accuracy']:.2f}%")
    print(f"  Correct: {result['correct']}/{result['total']}")
    print(f"  Parsing failures: {result['parsing_failures']}")
    print(f"  Log file: {result['log_file']}")
    
except Exception as e:
    print(f"\n✗ Error evaluating Qwen/Qwen3-30B-A3B: {e}")
